In [1]:
import math
import random
from collections.abc import Sequence
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as functional
from tqdm.auto import tqdm

from src.baselines import SoftAttentionUNet, StaticUNet
from src.dataloaders import get_dataloaders
from src.dynamic_unet import DynamicUNet
from src.losses import BCEDiceLoss
from src.metrics import (
    compute_confusion_counts,
    compute_segmentation_metrics,
    metrics_from_counts,
)
from src.transforms import TRAIN_MEAN, TRAIN_STD

RANDOM_SEED = 67
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
figure_directory = Path("results/figures")
figure_directory.mkdir(parents=True, exist_ok=True)
print(f"Evaluation device: {device}")

c:\Users\Anjney Lawaniya\Desktop\LifeLine\ML\Project\project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Evaluation device: cuda


In [2]:
static_model = StaticUNet().to(device)
attention_model = SoftAttentionUNet().to(device)
dynamic_model = DynamicUNet().to(device)

checkpoint_specs = (
    (static_model, Path("models/best_static_unet.pth")),
    (attention_model, Path("models/best_attention_unet.pth")),
    (dynamic_model, Path("models/best_dynamic_unet.pth")),
)
for model, checkpoint_path in checkpoint_specs:
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True,
    )
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()

_, val_loader, test_loader = get_dataloaders(batch_size=1, num_workers=4)

In [3]:
def convolution_gflops(
    spatial_size: int, input_channels: int, output_channels: int, kernel_size: int
) -> float:
    return (
        2.0
        * spatial_size
        * spatial_size
        * input_channels
        * output_channels
        * kernel_size
        * kernel_size
        / 1e9
    )


def decoder_transpose_gflops(
    spatial_size: int, input_channels: int, output_channels: int, kernel_size: int = 2
) -> float:
    input_spatial_size = spatial_size // 2
    return (
        2.0
        * input_spatial_size
        * input_spatial_size
        * input_channels
        * output_channels
        * kernel_size
        * kernel_size
        / 1e9
    )


def encoder_bottleneck_gflops(
    input_resolution: int = 512, input_channels: int = 3
) -> float:
    encoder_specs = (
        (input_resolution, input_channels, 64),
        (input_resolution // 2, 64, 128),
        (input_resolution // 4, 128, 256),
        (input_resolution // 8, 256, 512),
        (input_resolution // 16, 512, 1024),
    )
    total_cost = 0.0
    for spatial_size, in_ch, out_ch in encoder_specs:
        total_cost += convolution_gflops(spatial_size, in_ch, out_ch, 3)
        total_cost += convolution_gflops(spatial_size, out_ch, out_ch, 3)
    return total_cost


def static_decoder_gflops(input_resolution: int = 512) -> float:
    decoder_stage_specs = (
        (input_resolution // 8, 1024, 512),
        (input_resolution // 4, 512, 256),
        (input_resolution // 2, 256, 128),
        (input_resolution, 128, 64),
    )
    total_cost = 0.0
    for spatial_size, input_channels, output_channels in decoder_stage_specs:
        total_cost += decoder_transpose_gflops(spatial_size, input_channels, output_channels)
        total_cost += convolution_gflops(spatial_size, output_channels * 2, output_channels, 3)
        total_cost += convolution_gflops(spatial_size, output_channels, output_channels, 3)
    total_cost += convolution_gflops(input_resolution, 64, 1, 1)
    return total_cost


def attention_decoder_gflops(input_resolution: int = 512) -> float:
    decoder_stage_specs = (
        (input_resolution // 8, 1024, 512),
        (input_resolution // 4, 512, 256),
        (input_resolution // 2, 256, 128),
        (input_resolution, 128, 64),
    )
    total_cost = static_decoder_gflops(input_resolution)
    for spatial_size, _, output_channels in decoder_stage_specs:
        intermediate_channels = output_channels // 2
        total_cost += convolution_gflops(spatial_size, output_channels, intermediate_channels, 1)
        total_cost += convolution_gflops(spatial_size, output_channels, intermediate_channels, 1)
        total_cost += convolution_gflops(spatial_size, intermediate_channels, 1, 1)
    return total_cost


def dynamic_decoder_gflops(
    active_fractions: Sequence[float],
    dilated_active_fractions: Sequence[float] | None = None,
    conditional_low_entropy: bool = True,
) -> float:
    coarse_resolution = 64
    total_cost = decoder_transpose_gflops(coarse_resolution, 1024, 512)
    total_cost += convolution_gflops(coarse_resolution, 1024, 512, 3)
    total_cost += convolution_gflops(coarse_resolution, 512, 512, 3)

    gated_stage_specs = (
        (128, 512, 256),
        (256, 256, 128),
        (512, 128, 64),
    )
    halo_fractions = (
        list(active_fractions)
        if dilated_active_fractions is None
        else list(dilated_active_fractions)
    )
    for (spatial_size, input_channels, output_channels), raw_frac, halo_frac in zip(
        gated_stage_specs, active_fractions, halo_fractions
    ):
        total_cost += decoder_transpose_gflops(spatial_size, input_channels, output_channels)
        total_cost += convolution_gflops(spatial_size, output_channels, 1, 1)
        total_cost += convolution_gflops(spatial_size, output_channels * 2, output_channels, 1)
        single_conv3x3_cost = convolution_gflops(spatial_size, output_channels, output_channels, 3)
        background_projection_cost = convolution_gflops(
            spatial_size, output_channels, output_channels, 1
        )
        total_cost += (halo_frac + raw_frac) * single_conv3x3_cost
        low_entropy_multiplier = (1.0 - raw_frac) if conditional_low_entropy else 1.0
        total_cost += low_entropy_multiplier * background_projection_cost

    total_cost += convolution_gflops(512, 64, 1, 1)
    return total_cost


def count_parameters(model: nn.Module) -> dict[str, int]:
    total_params = sum(param.numel() for param in model.parameters())
    trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
    return {"total_params": int(total_params), "trainable_params": int(trainable_params)}


def accumulate_counts(total_counts: dict[str, int], logits: torch.Tensor, targets: torch.Tensor) -> None:
    batch_counts = compute_confusion_counts(logits.float(), targets.float())
    for count_name, count_value in batch_counts.items():
        total_counts[count_name] += count_value


def evaluate_fixed_model(model: nn.Module, data_loader) -> tuple[dict[str, int], dict[str, float]]:
    total_counts = {
        "true_positives": 0,
        "false_positives": 0,
        "false_negatives": 0,
        "true_negatives": 0,
    }
    model.eval()
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            accumulate_counts(total_counts, model(inputs), targets)
    return total_counts, metrics_from_counts(total_counts)


static_counts, static_metrics = evaluate_fixed_model(static_model, test_loader)
attention_counts, attention_metrics = evaluate_fixed_model(attention_model, test_loader)
static_parameters = count_parameters(static_model)
attention_parameters = count_parameters(attention_model)
encoder_gflops = encoder_bottleneck_gflops()
static_decoder_cost = static_decoder_gflops()
attention_decoder_cost = attention_decoder_gflops()

print(
    f"{'Model':<24} {'Test F1':>10} {'Test IoU':>10} {'Trainable Params':>18} "
    f"{'Total Params':>16} {'Decoder GFLOPs':>16} {'Total GFLOPs':>14}"
)
for display_name, metrics, parameters, decoder_cost in (
    ("Static U-Net", static_metrics, static_parameters, static_decoder_cost),
    ("Soft Attention U-Net", attention_metrics, attention_parameters, attention_decoder_cost),
):
    print(
        f"{display_name:<24} {metrics['f1_score']:>10.5f} {metrics['iou']:>10.5f} "
        f"{parameters['trainable_params']:>18,d} {parameters['total_params']:>16,d} "
        f"{decoder_cost:>16.3f} {encoder_gflops + decoder_cost:>14.3f}"
    )

Model                       Test F1   Test IoU   Trainable Params     Total Params   Decoder GFLOPs   Total GFLOPs
Static U-Net                0.61088    0.43976         31,037,633       31,037,633          249.142        385.339
Soft Attention U-Net        0.61728    0.44643         31,387,237       31,387,237          257.763        393.960


In [4]:
def compute_dilated_routing_mask(routing_mask: torch.Tensor) -> torch.Tensor:
    return functional.max_pool2d(
        routing_mask.float(), kernel_size=3, stride=1, padding=1
    ) > 0


def evaluate_threshold_sweep(
    model: DynamicUNet,
    data_loader,
    thresholds: Sequence[float],
    eval_device: str | torch.device,
) -> list[dict[str, float]]:
    original_threshold = model.threshold
    sweep_results: list[dict[str, float]] = []
    model.eval()
    try:
        for tau in thresholds:
            model.set_entropy_threshold(float(tau))
            counts = {
                "true_positives": 0,
                "false_positives": 0,
                "false_negatives": 0,
                "true_negatives": 0,
            }
            stage_raw_active = [0.0, 0.0, 0.0]
            stage_halo_active = [0.0, 0.0, 0.0]
            stage_total_pixels = [0, 0, 0]
            with torch.no_grad():
                for inputs, targets in data_loader:
                    inputs = inputs.to(eval_device, non_blocking=True)
                    targets = targets.to(eval_device, non_blocking=True)
                    logits, _, routing_masks_list = model(inputs, return_internals=True)
                    accumulate_counts(counts, logits, targets)
                    for stage_index, routing_mask in enumerate(routing_masks_list):
                        dilated_mask = compute_dilated_routing_mask(routing_mask)
                        stage_raw_active[stage_index] += float(routing_mask.sum().item())
                        stage_halo_active[stage_index] += float(dilated_mask.sum().item())
                        stage_total_pixels[stage_index] += routing_mask.numel()

            raw_fractions = [
                active / total if total else 0.0
                for active, total in zip(stage_raw_active, stage_total_pixels)
            ]
            halo_fractions = [
                active / total if total else 0.0
                for active, total in zip(stage_halo_active, stage_total_pixels)
            ]
            seg_metrics = metrics_from_counts(counts)
            sweep_results.append(
                {
                    "threshold": float(tau),
                    **seg_metrics,
                    "raw_active_percentage": 100.0 * float(np.mean(raw_fractions)),
                    "dilated_halo_active_percentage": 100.0 * float(np.mean(halo_fractions)),
                    "nominal_decoder_gflops": dynamic_decoder_gflops(raw_fractions),
                    "halo_adjusted_decoder_gflops": dynamic_decoder_gflops(
                        raw_fractions, halo_fractions
                    ),
                }
            )
    finally:
        model.set_entropy_threshold(original_threshold)
    return sweep_results


thresholds = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
validation_sweep = evaluate_threshold_sweep(dynamic_model, val_loader, thresholds, device)
selected_validation_threshold = max(
    validation_sweep,
    key=lambda row: row["f1_score"],
)["threshold"]
print(f"Validation-selected entropy threshold by micro-F1: {selected_validation_threshold:.1f}")
print(f"{'Validation tau':>15} {'Micro F1':>12} {'Micro IoU':>12}")
for result in validation_sweep:
    print(f"{result['threshold']:>15.1f} {result['f1_score']:>12.5f} {result['iou']:>12.5f}")

test_sweep = evaluate_threshold_sweep(dynamic_model, test_loader, thresholds, device)
print(
    f"\n{'Test tau':>9} {'Micro F1':>11} {'Micro IoU':>11} {'Precision':>11} {'Recall':>11} "
    f"{'Nominal GFLOPs':>16} {'Halo GFLOPs':>14} {'Raw Active %':>14} {'Halo Active %':>15}"
)
for result in test_sweep:
    print(
        f"{result['threshold']:>9.1f} {result['f1_score']:>11.5f} {result['iou']:>11.5f} "
        f"{result['precision']:>11.5f} {result['recall']:>11.5f} "
        f"{result['nominal_decoder_gflops']:>16.3f} "
        f"{result['halo_adjusted_decoder_gflops']:>14.3f} "
        f"{result['raw_active_percentage']:>13.2f}% "
        f"{result['dilated_halo_active_percentage']:>14.2f}%"
    )
print(
    "Dense dynamic decoder control at tau=0.0: "
    f"{test_sweep[0]['nominal_decoder_gflops']:.3f} GFLOPs; "
    f"pure 1x1 bypass at tau=1.0: {test_sweep[-1]['nominal_decoder_gflops']:.3f} GFLOPs."
)

Validation-selected entropy threshold by micro-F1: 0.5
 Validation tau     Micro F1    Micro IoU
            0.0      0.61877      0.44798
            0.1      0.62178      0.45114
            0.3      0.62143      0.45078
            0.5      0.62301      0.45244
            0.7      0.62204      0.45142
            0.9      0.61537      0.44443
            1.0      0.59569      0.42418

 Test tau    Micro F1   Micro IoU   Precision      Recall   Nominal GFLOPs    Halo GFLOPs   Raw Active %   Halo Active %
      0.0     0.61270     0.44165     0.58895     0.63845          204.103        204.103        100.00%         100.00%
      0.1     0.61462     0.44365     0.62055     0.60880          103.958        110.548          8.56%          19.93%
      0.3     0.61405     0.44305     0.62960     0.59925          100.494        106.918          5.40%          16.48%
      0.5     0.61561     0.44468     0.63223     0.59984           98.694        104.806          3.76%          14.30%
   

In [5]:
dynamic_f1_scores = [result["f1_score"] for result in test_sweep]
dynamic_gflops = [result["nominal_decoder_gflops"] for result in test_sweep]
dynamic_thresholds = [result["threshold"] for result in test_sweep]

fig, axis = plt.subplots(figsize=(8, 6), constrained_layout=True)
axis.scatter(
    [static_decoder_cost],
    [static_metrics["f1_score"]],
    color="red",
    marker="*",
    s=180,
    label="Static U-Net",
)
axis.scatter(
    [attention_decoder_cost],
    [attention_metrics["f1_score"]],
    color="blue",
    marker="s",
    s=80,
    label="Soft Attention U-Net",
)
axis.plot(
    dynamic_gflops,
    dynamic_f1_scores,
    color="green",
    marker="o",
    linestyle="-",
    label="Dynamic U-Net",
)
for tau, gflops, f1_score in zip(dynamic_thresholds, dynamic_gflops, dynamic_f1_scores):
    axis.annotate(
        rf"$\tau={tau:.1f}$",
        (gflops, f1_score),
        xytext=(5, 6),
        textcoords="offset points",
    )
axis.set_xlabel("Decoder GFLOPs")
axis.set_ylabel("F1 Score (Dice)")
axis.set_title("Decoder Compute vs. Road Segmentation Quality")
axis.grid(True, alpha=0.3)
axis.legend()
fig.savefig(figure_directory / "pareto_frontier.png", dpi=300, bbox_inches="tight")
plt.close(fig)

In [6]:
visual_inputs = None
visual_targets = None
for candidate_inputs, candidate_targets in test_loader:
    if bool(torch.any(candidate_targets > 0.5)):
        visual_inputs = candidate_inputs
        visual_targets = candidate_targets
        break
if visual_inputs is None:
    visual_inputs, visual_targets = next(iter(test_loader))

visual_inputs = visual_inputs.to(device)
visual_targets = visual_targets.to(device)
dynamic_model.set_entropy_threshold(0.5)
with torch.no_grad():
    static_logits = static_model(visual_inputs)
    dynamic_logits, auxiliary_logits_list, _ = dynamic_model(
        visual_inputs,
        return_internals=True,
    )
    _, entropy_map = dynamic_model.decoder1.entropy_gate(auxiliary_logits_list[-1])
    static_prediction = torch.sigmoid(static_logits) >= 0.5
    dynamic_prediction = torch.sigmoid(dynamic_logits) >= 0.5

mean_tensor = torch.tensor(TRAIN_MEAN, device=device).view(1, 3, 1, 1)
std_tensor = torch.tensor(TRAIN_STD, device=device).view(1, 3, 1, 1)
denormalized_image = (visual_inputs * std_tensor + mean_tensor).clamp(0.0, 1.0)
image_display = denormalized_image[0].permute(1, 2, 0).cpu().numpy()
target_display = visual_targets[0, 0].cpu().numpy()
static_display = static_prediction[0, 0].cpu().numpy()
dynamic_display = dynamic_prediction[0, 0].cpu().numpy()
entropy_display = entropy_map[0, 0].cpu().numpy()

fig, axes = plt.subplots(1, 5, figsize=(20, 4), constrained_layout=True)
axes[0].imshow(image_display)
axes[0].set_title("RGB Image")
axes[1].imshow(target_display, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Ground Truth")
axes[2].imshow(static_display, cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Static U-Net")
axes[3].imshow(dynamic_display, cmap="gray", vmin=0, vmax=1)
axes[3].set_title("Dynamic U-Net")
axes[4].imshow(entropy_display, cmap="magma", vmin=0, vmax=1)
axes[4].set_title("Shannon Entropy")
for axis in axes:
    axis.axis("off")
fig.savefig(figure_directory / "segmentation_overlay.png", dpi=300, bbox_inches="tight")
plt.close(fig)

In [7]:
from torchvision.models.segmentation import deeplabv3_resnet50

pixel_classifier = nn.Conv2d(3, 1, kernel_size=1).to(device)
deep_lab_probe = deeplabv3_resnet50(weights="DEFAULT").to(device)
for parameter in deep_lab_probe.parameters():
    parameter.requires_grad = False
deep_lab_probe.classifier[4] = nn.Conv2d(256, 1, kernel_size=1).to(device)
for parameter in deep_lab_probe.classifier[4].parameters():
    parameter.requires_grad = True

train_loader, _, _ = get_dataloaders(batch_size=4, num_workers=4)
probe_loss = BCEDiceLoss()
pixel_optimizer = torch.optim.AdamW(pixel_classifier.parameters(), lr=1e-3)
deep_lab_optimizer = torch.optim.AdamW(
    deep_lab_probe.classifier[4].parameters(),
    lr=1e-3,
)
pixel_scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
deep_lab_scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
project_mean = torch.tensor(TRAIN_MEAN, device=device).view(1, 3, 1, 1)
project_std = torch.tensor(TRAIN_STD, device=device).view(1, 3, 1, 1)
imagenet_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

pixel_classifier.train()
deep_lab_probe.eval()
for inputs, targets in tqdm(train_loader, desc="One epoch: lightweight probes", leave=False):
    inputs = inputs.to(device, non_blocking=True)
    targets = targets.to(device, non_blocking=True)
    raw_rgb = (inputs * project_std + project_mean).clamp(0.0, 1.0)
    imagenet_inputs = (raw_rgb - imagenet_mean) / imagenet_std

    pixel_optimizer.zero_grad(set_to_none=True)
    with torch.amp.autocast(device_type=torch.device(device).type, enabled=device == "cuda"):
        pixel_loss = probe_loss(pixel_classifier(inputs), targets)
    pixel_scaler.scale(pixel_loss).backward()
    pixel_scaler.step(pixel_optimizer)
    pixel_scaler.update()

    deep_lab_optimizer.zero_grad(set_to_none=True)
    with torch.amp.autocast(device_type=torch.device(device).type, enabled=device == "cuda"):
        deep_lab_loss = probe_loss(deep_lab_probe(imagenet_inputs)["out"], targets)
    deep_lab_scaler.scale(deep_lab_loss).backward()
    deep_lab_scaler.step(deep_lab_optimizer)
    deep_lab_scaler.update()

pixel_classifier.eval()
deep_lab_probe.eval()
probe_counts = {
    "pixel": {"true_positives": 0, "false_positives": 0, "false_negatives": 0, "true_negatives": 0},
    "deeplab": {"true_positives": 0, "false_positives": 0, "false_negatives": 0, "true_negatives": 0},
}
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        raw_rgb = (inputs * project_std + project_mean).clamp(0.0, 1.0)
        imagenet_inputs = (raw_rgb - imagenet_mean) / imagenet_std
        accumulate_counts(probe_counts["pixel"], pixel_classifier(inputs), targets)
        accumulate_counts(
            probe_counts["deeplab"],
            deep_lab_probe(imagenet_inputs)["out"],
            targets,
        )


def measure_convolution_gflops(model: nn.Module, sample_tensor: torch.Tensor) -> float:
    measured_flops = {"value": 0.0}
    hook_handles = []

    def count_convolution(module, module_inputs, module_output):
        if isinstance(module_output, torch.Tensor):
            output_height, output_width = module_output.shape[-2:]
            operations = (
                2
                * output_height
                * output_width
                * module.in_channels
                * module.out_channels
                * module.kernel_size[0]
                * module.kernel_size[1]
                / module.groups
            )
            measured_flops["value"] += operations / 1e9

    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            hook_handles.append(module.register_forward_hook(count_convolution))
    with torch.no_grad():
        model(sample_tensor)
    for hook_handle in hook_handles:
        hook_handle.remove()
    return measured_flops["value"]


raw_sample = (visual_inputs * project_std + project_mean).clamp(0.0, 1.0)
imagenet_sample = (raw_sample - imagenet_mean) / imagenet_std
deep_lab_total_gflops = measure_convolution_gflops(deep_lab_probe, imagenet_sample)
pixel_head_gflops = convolution_gflops(512, 3, 1, 1)
deep_lab_head_gflops = convolution_gflops(64, 256, 1, 1)
static_total_gflops = encoder_gflops + static_decoder_cost
attention_total_gflops = encoder_gflops + attention_decoder_cost
dynamic_half_result = next(result for result in test_sweep if result["threshold"] == 0.5)
dynamic_decoder_cost = dynamic_half_result["nominal_decoder_gflops"]
dynamic_total_gflops = encoder_gflops + dynamic_decoder_cost

comparison_metrics = {
    "pixel": metrics_from_counts(probe_counts["pixel"]),
    "deeplab": metrics_from_counts(probe_counts["deeplab"]),
    "static": static_metrics,
    "attention": attention_metrics,
    "dynamic": dynamic_half_result,
}
comparison_parameters = {
    "pixel": count_parameters(pixel_classifier),
    "deeplab": count_parameters(deep_lab_probe),
    "static": static_parameters,
    "attention": attention_parameters,
    "dynamic": count_parameters(dynamic_model),
}
comparison_rows = (
    ("1x1 Pixel Classifier (1-epoch probe)", "pixel", pixel_head_gflops, pixel_head_gflops),
    ("Frozen DeepLabV3-ResNet50 Probe (1 epoch)", "deeplab", deep_lab_head_gflops, deep_lab_total_gflops),
    ("Static U-Net", "static", static_decoder_cost, static_total_gflops),
    ("Soft Attention U-Net", "attention", attention_decoder_cost, attention_total_gflops),
    ("Dynamic U-Net (tau=0.5)", "dynamic", dynamic_decoder_cost, dynamic_total_gflops),
)
print(
    f"{'Model':<48} {'Test F1':>9} {'Test IoU':>9} {'Trainable Params':>18} "
    f"{'Total Params':>15} {'Head/Decoder GFLOPs':>21} {'End-to-End GFLOPs':>19}"
)
for model_label, model_key, head_cost, end_to_end_cost in comparison_rows:
    metrics = comparison_metrics[model_key]
    parameters = comparison_parameters[model_key]
    print(
        f"{model_label:<48} {metrics['f1_score']:>9.5f} {metrics['iou']:>9.5f} "
        f"{parameters['trainable_params']:>18,d} {parameters['total_params']:>15,d} "
        f"{head_cost:>21.3f} {end_to_end_cost:>19.3f}"
    )

Model                                              Test F1  Test IoU   Trainable Params    Total Params   Head/Decoder GFLOPs   End-to-End GFLOPs
1x1 Pixel Classifier (1-epoch probe)               0.00000   0.00000                  4               4                 0.002               0.002
Frozen DeepLabV3-ResNet50 Probe (1 epoch)          0.00000   0.00000                257      41,998,934                 0.002             346.488
Static U-Net                                       0.61088   0.43976         31,037,633      31,037,633               249.142             385.339
Soft Attention U-Net                               0.61728   0.44643         31,387,237      31,387,237               257.763             393.960
Dynamic U-Net (tau=0.5)                            0.61561   0.44468         30,523,780      30,523,780                98.694             234.892


In [8]:
def evaluate_routing_policies(
    model: DynamicUNet,
    data_loader,
    eval_device: str | torch.device,
    threshold: float = 0.5,
    seed: int = 67,
) -> list[tuple[str, dict[str, float]]]:
    original_threshold = model.threshold
    routing_state = {"policy": "entropy", "samples_seen": [0, 0, 0]}
    hook_handles = []

    def make_routing_hook(stage_index: int):
        def hook(module, module_inputs, module_outputs):
            routing_mask, entropy_map = module_outputs
            policy = routing_state["policy"]
            if policy not in ("random", "inverted"):
                return routing_mask, entropy_map

            batch_size, _, height, width = routing_mask.shape
            replacement_mask = torch.zeros_like(routing_mask)
            for batch_index in range(batch_size):
                active_count = int(routing_mask[batch_index, 0].sum().item())
                if active_count <= 0:
                    continue
                flattened_mask = replacement_mask[batch_index, 0].reshape(-1)
                if policy == "random":
                    sample_idx = routing_state["samples_seen"][stage_index] + batch_index
                    generator = torch.Generator(device=routing_mask.device)
                    generator.manual_seed(seed + sample_idx * 31 + stage_index)
                    selected_indices = torch.randperm(
                        height * width,
                        generator=generator,
                        device=routing_mask.device,
                    )[:active_count]
                else:
                    flattened_entropy = entropy_map[batch_index, 0].reshape(-1)
                    selected_indices = torch.topk(
                        flattened_entropy,
                        k=active_count,
                        largest=False,
                    ).indices
                flattened_mask[selected_indices] = 1.0
            routing_state["samples_seen"][stage_index] += batch_size
            return replacement_mask, entropy_map

        return hook

    for stage_idx, stage_module in enumerate((model.decoder3, model.decoder2, model.decoder1)):
        hook_handles.append(
            stage_module.entropy_gate.register_forward_hook(make_routing_hook(stage_idx))
        )

    policy_specs = (
        ("All-Active Dynamic U-Net (tau=0.0, 100% Active 3x3)", "entropy", 0.0),
        (f"Shannon Entropy Routing (tau={threshold:.1f})", "entropy", threshold),
        ("Random Bernoulli Routing (Per-Tile Matched Budget)", "random", threshold),
        (
            "Inverted Confidence Routing (Bottom-k Lowest Entropy, Per-Tile Matched Budget)",
            "inverted",
            threshold,
        ),
        ("Pure 1x1 Bypass (tau=1.0, 0% Active 3x3)", "entropy", 1.0),
    )

    results: list[tuple[str, dict[str, float]]] = []
    model.eval()
    try:
        for display_name, policy_name, tau in policy_specs:
            routing_state["policy"] = policy_name
            routing_state["samples_seen"] = [0, 0, 0]
            model.set_entropy_threshold(tau)
            counts = {
                "true_positives": 0,
                "false_positives": 0,
                "false_negatives": 0,
                "true_negatives": 0,
            }
            stage_raw_active = [0.0, 0.0, 0.0]
            stage_halo_active = [0.0, 0.0, 0.0]
            stage_total_pixels = [0, 0, 0]
            with torch.no_grad():
                for inputs, targets in data_loader:
                    inputs = inputs.to(eval_device, non_blocking=True)
                    targets = targets.to(eval_device, non_blocking=True)
                    logits, _, routing_masks = model(inputs, return_internals=True)
                    accumulate_counts(counts, logits, targets)
                    for stage_index, routing_mask in enumerate(routing_masks):
                        dilated_mask = compute_dilated_routing_mask(routing_mask)
                        stage_raw_active[stage_index] += float(routing_mask.sum().item())
                        stage_halo_active[stage_index] += float(dilated_mask.sum().item())
                        stage_total_pixels[stage_index] += routing_mask.numel()

            raw_fractions = [
                active / total if total else 0.0
                for active, total in zip(stage_raw_active, stage_total_pixels)
            ]
            halo_fractions = [
                active / total if total else 0.0
                for active, total in zip(stage_halo_active, stage_total_pixels)
            ]
            seg_metrics = metrics_from_counts(counts)
            results.append(
                (
                    display_name,
                    {
                        **seg_metrics,
                        "raw_active_percentage": 100.0 * float(np.mean(raw_fractions)),
                        "dilated_halo_active_percentage": 100.0 * float(np.mean(halo_fractions)),
                        "nominal_decoder_gflops": dynamic_decoder_gflops(raw_fractions),
                        "halo_adjusted_decoder_gflops": dynamic_decoder_gflops(
                            raw_fractions, halo_fractions
                        ),
                    },
                )
            )
    finally:
        for handle in hook_handles:
            handle.remove()
        model.set_entropy_threshold(original_threshold)

    return results


routing_policy_results = evaluate_routing_policies(
    dynamic_model,
    test_loader,
    device,
    threshold=0.5,
    seed=67,
)
print(
    f"{'Routing Policy':<82} {'F1':>9} {'IoU':>9} {'Raw Active %':>14} "
    f"{'Halo Active %':>15} {'Nominal GFLOPs':>16} {'Halo GFLOPs':>14}"
)
for policy_name, policy_metrics in routing_policy_results:
    print(
        f"{policy_name:<82} {policy_metrics['f1_score']:>9.5f} "
        f"{policy_metrics['iou']:>9.5f} "
        f"{policy_metrics['raw_active_percentage']:>13.2f}% "
        f"{policy_metrics['dilated_halo_active_percentage']:>14.2f}% "
        f"{policy_metrics['nominal_decoder_gflops']:>16.3f} "
        f"{policy_metrics['halo_adjusted_decoder_gflops']:>14.3f}"
    )

Routing Policy                                                                            F1       IoU   Raw Active %   Halo Active %   Nominal GFLOPs    Halo GFLOPs
All-Active Dynamic U-Net (tau=0.0, 100% Active 3x3)                                  0.61270   0.44165        100.00%         100.00%          204.103        204.103
Shannon Entropy Routing (tau=0.5)                                                    0.61561   0.44468          3.76%          14.30%           98.694        104.806
Random Bernoulli Routing (Per-Tile Matched Budget)                                   0.59264   0.42110          4.28%          29.73%           99.267        114.023
Inverted Confidence Routing (Bottom-k Lowest Entropy, Per-Tile Matched Budget)       0.58831   0.41674          4.33%          11.88%           99.324        103.701
Pure 1x1 Bypass (tau=1.0, 0% Active 3x3)                                             0.59032   0.41876          0.00%           0.00%           94.582         94.582


In [9]:
try:
    from scipy.ndimage import distance_transform_edt
except ImportError:
    distance_transform_edt = None


def extract_binary_boundary(binary_mask: torch.Tensor, radius: int = 1) -> torch.Tensor:
    binary_mask = binary_mask.float()
    kernel_size = 2 * radius + 1
    padded_mask = functional.pad(
        binary_mask,
        (radius, radius, radius, radius),
        mode="replicate",
    )
    dilated_mask = functional.max_pool2d(padded_mask, kernel_size=kernel_size, stride=1)
    eroded_mask = -functional.max_pool2d(-padded_mask, kernel_size=kernel_size, stride=1)
    return dilated_mask > eroded_mask


def compute_boundary_f1(
    logits: torch.Tensor,
    targets: torch.Tensor,
    boundary_radius: int = 1,
    tolerance_radius: int = 2,
) -> float:
    predicted_mask = torch.sigmoid(logits) >= 0.5
    target_mask = targets >= 0.5
    predicted_boundary = extract_binary_boundary(predicted_mask, radius=boundary_radius)
    target_boundary = extract_binary_boundary(target_mask, radius=boundary_radius)
    kernel_size = 2 * tolerance_radius + 1
    predicted_tolerance = (
        functional.max_pool2d(
            predicted_boundary.float(),
            kernel_size=kernel_size,
            stride=1,
            padding=tolerance_radius,
        )
        > 0
    )
    target_tolerance = (
        functional.max_pool2d(
            target_boundary.float(),
            kernel_size=kernel_size,
            stride=1,
            padding=tolerance_radius,
        )
        > 0
    )
    matched_predicted = int((predicted_boundary & target_tolerance).sum().item())
    matched_target = int((target_boundary & predicted_tolerance).sum().item())
    predicted_count = int(predicted_boundary.sum().item())
    target_count = int(target_boundary.sum().item())
    if predicted_count == 0 and target_count == 0:
        return 1.0
    precision = matched_predicted / predicted_count if predicted_count else 0.0
    recall = matched_target / target_count if target_count else 0.0
    return 2.0 * precision * recall / (precision + recall) if (precision + recall) else 0.0


def compute_hd95(
    logits: torch.Tensor,
    targets: torch.Tensor,
    boundary_radius: int = 1,
) -> tuple[float, bool]:
    predicted_boundary = extract_binary_boundary(
        torch.sigmoid(logits) >= 0.5, radius=boundary_radius
    )[0, 0]
    target_boundary = extract_binary_boundary(
        targets >= 0.5, radius=boundary_radius
    )[0, 0]
    predicted_coordinates = torch.nonzero(predicted_boundary, as_tuple=False).float()
    target_coordinates = torch.nonzero(target_boundary, as_tuple=False).float()
    if predicted_coordinates.numel() == 0 and target_coordinates.numel() == 0:
        return 0.0, True
    if predicted_coordinates.numel() == 0 or target_coordinates.numel() == 0:
        height, width = predicted_boundary.shape
        return float((height * height + width * width) ** 0.5), False

    if distance_transform_edt is not None:
        predicted_array = predicted_boundary.cpu().numpy()
        target_array = target_boundary.cpu().numpy()
        distances_to_target = distance_transform_edt(~target_array)[predicted_array]
        distances_to_prediction = distance_transform_edt(~predicted_array)[target_array]
        symmetric_distances = np.concatenate((distances_to_target, distances_to_prediction))
        return float(np.percentile(symmetric_distances, 95)), True

    def directed_nearest_distances(
        source_coordinates: torch.Tensor, destination_coordinates: torch.Tensor
    ) -> torch.Tensor:
        distance_chunks = []
        for chunk_start in range(0, source_coordinates.shape[0], 1024):
            coordinate_chunk = source_coordinates[chunk_start : chunk_start + 1024]
            pairwise_distances = torch.cdist(coordinate_chunk, destination_coordinates)
            distance_chunks.append(pairwise_distances.min(dim=1).values)
        return torch.cat(distance_chunks)

    forward_distances = directed_nearest_distances(predicted_coordinates, target_coordinates)
    reverse_distances = directed_nearest_distances(target_coordinates, predicted_coordinates)
    hd95_val = float(
        torch.quantile(torch.cat((forward_distances, reverse_distances)), 0.95).item()
    )
    return hd95_val, True


def bootstrap_confidence_interval(
    values: Sequence[float],
    num_bootstrap: int = 1000,
    confidence: float = 0.95,
    seed: int = 67,
) -> tuple[float, float, float]:
    array = np.asarray(values, dtype=np.float64)
    if array.size == 0:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, array.size, size=(num_bootstrap, array.size))
    bootstrap_means = array[indices].mean(axis=1)
    alpha = (1.0 - confidence) / 2.0
    ci_lower, ci_upper = np.quantile(bootstrap_means, [alpha, 1.0 - alpha])
    return float(array.mean()), float(ci_lower), float(ci_upper)


dynamic_model.set_entropy_threshold(0.5)
per_image_records = []
with torch.no_grad():
    for inputs, targets in tqdm(test_loader, desc="Per-tile topology metrics"):
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        static_logits = static_model(inputs)
        attention_logits = attention_model(inputs)
        dynamic_logits, _, routing_masks = dynamic_model(inputs, return_internals=True)
        dynamic_active_percentage = 100.0 * float(
            torch.stack([routing_mask.float().mean() for routing_mask in routing_masks]).mean().item()
        )
        record = {
            "road_density_percentage": 100.0 * float((targets >= 0.5).float().mean().item()),
            "dynamic_active_percentage": dynamic_active_percentage,
        }
        for model_name, logits in (
            ("static", static_logits),
            ("attention", attention_logits),
            ("dynamic", dynamic_logits),
        ):
            metrics = compute_segmentation_metrics(logits, targets)
            hd95, hd95_valid = compute_hd95(logits, targets, boundary_radius=1)
            record[f"{model_name}_f1"] = metrics["f1_score"]
            record[f"{model_name}_iou"] = metrics["iou"]
            record[f"{model_name}_boundary_f1"] = compute_boundary_f1(
                logits,
                targets,
                boundary_radius=1,
                tolerance_radius=2,
            )
            record[f"{model_name}_hd95"] = hd95
            record[f"{model_name}_hd95_valid"] = hd95_valid
        per_image_records.append(record)

median_road_density = float(
    np.median([record["road_density_percentage"] for record in per_image_records])
)
for record in per_image_records:
    record["terrain_subgroup"] = (
        "Low Road-Density Tiles (<= Median, Rural/Coastal Proxy)"
        if record["road_density_percentage"] <= median_road_density
        else "High Road-Density Tiles (> Median, Urban Proxy)"
    )

subgroup_names = (
    "Low Road-Density Tiles (<= Median, Rural/Coastal Proxy)",
    "High Road-Density Tiles (> Median, Urban Proxy)",
)
subgroup_metric_names = (
    "static_f1", "static_iou", "static_boundary_f1", "static_hd95",
    "attention_f1", "attention_iou", "attention_boundary_f1", "attention_hd95",
    "dynamic_f1", "dynamic_iou", "dynamic_boundary_f1", "dynamic_hd95",
)
for subgroup_name in subgroup_names:
    subgroup_records = [
        record for record in per_image_records if record["terrain_subgroup"] == subgroup_name
    ]
    print(f"\n{subgroup_name}: n={len(subgroup_records)}")
    for metric_name in subgroup_metric_names:
        if metric_name.endswith("hd95"):
            model_name = metric_name.split("_")[0]
            valid_values = [
                record[metric_name]
                for record in subgroup_records
                if record[f"{model_name}_hd95_valid"]
            ]
            mean_hd95 = float(np.mean(valid_values)) if valid_values else float("nan")
            median_hd95 = float(np.median(valid_values)) if valid_values else float("nan")
            print(
                f"{metric_name:<23} Mean HD95={mean_hd95:.3f}, "
                f"Median HD95={median_hd95:.3f}, valid n={len(valid_values)}"
            )
        else:
            mean_value, ci_lower, ci_upper = bootstrap_confidence_interval(
                [record[metric_name] for record in subgroup_records],
                num_bootstrap=1000,
                confidence=0.95,
                seed=67,
            )
            print(
                f"{metric_name:<23} Mean={mean_value:.5f}, "
                f"95% Bootstrap CI=[{ci_lower:.5f}, {ci_upper:.5f}]"
            )

road_density_values = np.asarray(
    [record["road_density_percentage"] for record in per_image_records]
)
dynamic_active_values = np.asarray(
    [record["dynamic_active_percentage"] for record in per_image_records]
)
fig, axis = plt.subplots(figsize=(8, 6), constrained_layout=True)
axis.scatter(
    road_density_values,
    dynamic_active_values,
    alpha=0.75,
    color="teal",
    label="Test tiles",
)
if road_density_values.size > 1 and np.ptp(road_density_values) > 0:
    trend_coefficients = np.polyfit(road_density_values, dynamic_active_values, deg=1)
    trend_x = np.linspace(road_density_values.min(), road_density_values.max(), 100)
    axis.plot(
        trend_x,
        np.polyval(trend_coefficients, trend_x),
        color="darkorange",
        linewidth=2,
        label="Linear trend",
    )
axis.set_xlabel("Per-Tile Ground-Truth Road Density (%)")
axis.set_ylabel("Dynamic U-Net Active 3x3 Compute (%)")
axis.set_title("Adaptive Decoder Compute by Road Density")
axis.grid(True, alpha=0.3)
axis.legend()
fig.savefig(
    figure_directory / "adaptive_compute_scatter.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)

Per-tile topology metrics: 100%|██████████| 167/167 [00:41<00:00,  3.98it/s]



Low Road-Density Tiles (<= Median, Rural/Coastal Proxy): n=84
static_f1               Mean=0.48158, 95% Bootstrap CI=[0.43336, 0.52446]
static_iou              Mean=0.34118, 95% Bootstrap CI=[0.30436, 0.37443]
static_boundary_f1      Mean=0.65760, 95% Bootstrap CI=[0.59213, 0.71267]
static_hd95             Mean HD95=122.761, Median HD95=69.635, valid n=81
attention_f1            Mean=0.47996, 95% Bootstrap CI=[0.43385, 0.52037]
attention_iou           Mean=0.33826, 95% Bootstrap CI=[0.30301, 0.37020]
attention_boundary_f1   Mean=0.65131, 95% Bootstrap CI=[0.59004, 0.70558]
attention_hd95          Mean HD95=111.916, Median HD95=66.578, valid n=80
dynamic_f1              Mean=0.49584, 95% Bootstrap CI=[0.44625, 0.53977]
dynamic_iou             Mean=0.35569, 95% Bootstrap CI=[0.31714, 0.39146]
dynamic_boundary_f1     Mean=0.66676, 95% Bootstrap CI=[0.60283, 0.72320]
dynamic_hd95            Mean HD95=114.206, Median HD95=62.828, valid n=80

High Road-Density Tiles (> Median, Urban Proxy):